<a href="https://colab.research.google.com/github/Hadi6618/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection/blob/First-Branch/MULDE_Training_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MULDE Training Pipeline
This notebook clones your modified MULDE repository, mounts Google Drive to access the Hiera-L features, and trains the MULDE anomaly detection model.

**Features:**
- Checkpoints saved to Google Drive every 50 epochs (survives Colab crashes)
- Automatically resumes from the last checkpoint
- AUC-ROC scores printed directly to terminal (no TensorBoard needed)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!tar -xzvf "/content/drive/MyDrive/shanghaitech.tar.gz" -C "/content/"

Streaming output truncated to the last 5000 lines.
shanghaitech/testing/frames/08_0077/079.jpg
shanghaitech/testing/frames/08_0077/269.jpg
shanghaitech/testing/frames/08_0077/113.jpg
shanghaitech/testing/frames/08_0077/149.jpg
shanghaitech/testing/frames/08_0077/084.jpg
shanghaitech/testing/frames/08_0077/338.jpg
shanghaitech/testing/frames/08_0077/028.jpg
shanghaitech/testing/frames/08_0077/402.jpg
shanghaitech/testing/frames/08_0077/169.jpg
shanghaitech/testing/frames/08_0077/288.jpg
shanghaitech/testing/frames/08_0077/222.jpg
shanghaitech/testing/frames/08_0077/283.jpg
shanghaitech/testing/frames/08_0077/126.jpg
shanghaitech/testing/frames/08_0077/122.jpg
shanghaitech/testing/frames/08_0077/455.jpg
shanghaitech/testing/frames/08_0077/378.jpg
shanghaitech/testing/frames/08_0077/270.jpg
shanghaitech/testing/frames/08_0077/177.jpg
shanghaitech/testing/frames/08_0077/305.jpg
shanghaitech/testing/frames/08_0077/210.jpg
shanghaitech/testing/frames/08_0077/428.jpg
shanghaitech/testing/fram

In [3]:
import os
repo_dir = '/content/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection'

if os.path.exists(repo_dir):
    print('Repository already cloned. Pulling latest changes...')
    !cd {repo_dir} && git pull
else:
    !git clone -b First-Branch https://github.com/Hadi6618/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection.git

%cd {repo_dir}

Cloning into 'MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 77 (delta 36), reused 43 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 377.78 KiB | 2.05 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection


## Setup Ground Truth Labels
The test phase requires the frame-level ground truth `.npy` labels for the ShanghaiTech dataset.

Modify the path below if your labels are stored in a different location.

In [4]:
# Uncomment and modify if you need to extract the labels from a zip on Drive:
# !unzip -q "/content/drive/MyDrive/path_to_shanghaitech.zip" -d "/content/shanghaitech"

import os
label_dir = '/content/shanghaitech/testing/test_frame_mask'
if os.path.isdir(label_dir):
    n_labels = len([f for f in os.listdir(label_dir) if f.endswith('.npy')])
    print(f'Found {n_labels} label files in {label_dir}')
else:
    print(f'WARNING: Label directory not found at {label_dir}')
    print('Please extract or mount the ShanghaiTech labels before training.')

Found 107 label files in /content/shanghaitech/testing/test_frame_mask


## Run MULDE Training
Parameters match the CVPR 2024 paper for **frame-centric** ShanghaiTech.

Key paper parameters:
- `--lr 1e-4` (paper specifies 1e-4 for frame-centric, 5e-4 for object-centric)
- `--beta 0.1` (regularization, paper Section 3.4)
- `--gmm` (GMM aggregation with 1,3,5 components, paper Section 3.3)
- `--L 16` (16 evenly spaced noise scales, paper Section 7.3)

**Checkpoints** saved to Google Drive every 50 epochs.
**Auto-resume**: If Colab crashes, just re-run this cell.

In [ ]:
!python main.py \
  --experiment_name "HieraL_ShanghaiTech" \
  --train_dir "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train" \
  --test_dir "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test" \
  --test_labels_dir "/content/shanghaitech/testing/test_frame_mask" \
  --stats_path "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train_feature_stats.npz" \
  --checkpoint_dir "/content/drive/MyDrive/MULDE/checkpoints" \
  --batch_size 2048 \
  --epochs 1000 \
  --lr 1e-4 \
  --beta 0.1 \
  --L 16 \
  --sigma_low 1e-3 \
  --sigma_high 1. \
  --units 4096 4096 \
  --eval_freq 50 \
  --checkpoint_freq 50 \
  --gmm

2026-05-30 01:52:46.682392: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading training features from /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train...
Loaded training data: (274515, 1152)
Loading testing features from /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test...
Loaded testing data: (40791, 1152)
Loaded testing labels: (40791,) (Anomalies: 17326)
Loading precomputed statistics from /content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train_feature_stats.npz...
No checkpoint found. Starting training from scratch.
Train Epoch 0: 100% 135/135 [00:33<00:00,  4.04it/s]
Calculate noise free log-densities/score norms acros